# 05 · Cadena de custodia criptográfica conforme a ISO/IEC 27037

**Proyecto de Investigación II — Protocolo experimental (secciones 7.1 y 8 del informe; hipótesis H5 y H6)**

Implementa y verifica el módulo de custodia de extremo a extremo que protege los cuatro flujos de datos del sistema: (i) fotografías de entrada, (ii) **informes de verificación de rajaduras recibidos por correo electrónico** (el dato de referencia), (iii) conjuntos de calibración y (iv) dictámenes generados.

| Mecanismo | Norma | Verificación en este notebook |
|---|---|---|
| Huella SHA-256 en la ingesta | RFC 6234 | alteración de 1 píxel / 1 carácter → huella distinta |
| Sello temporal RFC 3161 | RFC 3161 | solicitud DER correcta + TSA real (con respaldo local) + ventana de validez |
| Firma digital ECDSA P-256 | FIPS 186-5 | firma de la raíz de Merkle por expediente, verificable por terceros |
| Árbol de Merkle por expediente | Merkle (1978) | raíz única + prueba de inclusión logarítmica |
| Manifiesto de procedencia | C2PA 2.5 | manifiesto firmado (capa de interoperabilidad) |

Pruebas de **manipulación activa** (T1–T5) con control negativo, y medición del **costo operativo** de H6: latencia de sellado < 50 ms/expediente y sobrecarga < 1 MB/expediente.

> El notebook se ejecuta con un **expediente de demostración autogenerado** (fotografías sintéticas + informe de verificación simulado). En producción, las mismas funciones se aplican a los archivos reales de `Drive/grietas/`.


In [1]:
# ── 1. Entorno y utilidades criptográficas base ───────────────────────────────
import hashlib, json, time, base64, os, io, math, random, struct
from pathlib import Path
from datetime import datetime, timedelta, timezone

EN_COLAB = 'google.colab' in str(get_ipython())
if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/grietas')
else:
    BASE = Path('grietas_local')
    BASE.mkdir(parents=True, exist_ok=True)
INFORME = BASE / 'informe'
INFORME.mkdir(parents=True, exist_ok=True)
DEMO = BASE / 'expediente_demo'
DEMO.mkdir(parents=True, exist_ok=True)

from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives import hashes
from cryptography.exceptions import InvalidSignature

CLAVE_ECDSA = ec.generate_private_key(ec.SECP256R1())       # P-256 (FIPS 186-5)
print('Entorno Colab:', EN_COLAB)
print('Carpeta del expediente demo:', DEMO)
print('Clave ECDSA P-256 generada (demo). En producción, clave del perito emitida '
      'por autoridad certificadora.')


Mounted at /content/drive
Entorno Colab: True
Carpeta del expediente demo: /content/drive/MyDrive/grietas/expediente_demo
Clave ECDSA P-256 generada (demo). En producción, clave del perito emitida por autoridad certificadora.


In [2]:
# ── 2. Expediente de demostración: fotografías + informe de verificación ───────
import numpy as np
from PIL import Image as PILImage

rng = np.random.default_rng(42)
def foto_sintetica(nombre, semilla):
    # Imagen gris con una 'fisura' (línea oscura aleatoria): simula captura de campo
    g = 150 + rng.integers(-12, 12, size=(256, 256))
    x = np.linspace(10, 246, 160)
    y = 128 + 22 * np.sin((x + semilla) / 37.0) + rng.integers(-3, 3, size=160)
    for xi, yi in zip(x.astype(int), y.astype(int)):
        g[max(0, yi - 2):yi + 3, max(0, xi - 1):xi + 2] = 45
    ruta = DEMO / nombre
    PILImage.fromarray(g.astype('uint8')).save(ruta, format='PNG')
    return ruta

FOTOS = [foto_sintetica('foto_%d.png' % k, 13 * k) for k in range(3)]

INFORME_TXT = DEMO / 'informe_verificacionrajadura.txt'
INFORME_TXT.write_text(
    'AQUATECH — INFORME DE VERIFICACIÓN DE RAJADURAS\n'
    'Expediente EXP-2026-001 · Elemento: losa de concreto armado\n'
    'EXPOSICION declarada: humedad\n'
    'Fisura F01 · lectura de regla de ancho de grietas: 0,28 mm · clase: fina (LEVE) · '
    'ACI 224R-01 (w_tol 0,30 mm): NO_EXCEDE\n'
    'Fisura F02 · lectura de regla: 0,35 mm · clase: fina (LEVE) · '
    'ACI 224R-01 (w_tol 0,30 mm): EXCEDE\n'
    'Fisura F03 · lectura de regla: 1,60 mm · clase: media (MODERADA) · '
    'ACI 224R-01: EXCEDE\n'
    'Perito verificador: [nombre del perito] · remitido por correo electrónico.\n',
    encoding='utf-8')

for p in FOTOS + [INFORME_TXT]:
    print('%-34s %7.1f KiB' % (p.name, p.stat().st_size / 1024))


foto_0.png                            42.7 KiB
foto_1.png                            42.7 KiB
foto_2.png                            42.7 KiB
informe_verificacionrajadura.txt       0.5 KiB


In [3]:
# ── 3. Huella SHA-256 (RFC 6234) en la ingesta ─────────────────────────────────
def huella_sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, 'rb') as f:
        for bloque in iter(lambda: f.read(1 << 20), b''):
            h.update(bloque)
    return h.hexdigest()

def huella_bytes(b):
    return hashlib.sha256(b).hexdigest()

HUELLAS = {p.name: huella_sha256(p) for p in FOTOS + [INFORME_TXT]}
for n, h in HUELLAS.items():
    print('%-34s %s' % (n, h[:24] + '…'))
print()
# Demostración de sensibilidad: alterar UN byte de una copia
copia = DEMO / 'prueba_1bit.png'
copia.write_bytes((DEMO / 'foto_0.png').read_bytes())
b = bytearray(copia.read_bytes())
b[len(b) // 2] ^= 0x01                       # un solo bit
copia.write_bytes(bytes(b))
print('Huella original :', HUELLAS['foto_0.png'][:32] + '…')
print('Huella alterada :', huella_sha256(copia)[:32] + '…')
print('¿Detección de la alteración de 1 bit?:',
      'SÍ — huella distinta' if huella_sha256(copia) != HUELLAS['foto_0.png'] else 'NO')
copia.unlink()


foto_0.png                         fd892271d8ddf918c23df705…
foto_1.png                         7d6061ce2b2ffb764900964a…
foto_2.png                         6155ecddc4061f271b1d27dd…
informe_verificacionrajadura.txt   81867735ac43f40a562768ee…

Huella original : fd892271d8ddf918c23df7058947bfe8…
Huella alterada : 1fa16ffd47df61b9e59e3253e4665123…
¿Detección de la alteración de 1 bit?: SÍ — huella distinta


In [4]:
# ── 4. Sello temporal RFC 3161 (solicitud DER + TSA, con respaldo local) ──────
def der_longitud(n):
    if n < 0x80:
        return bytes([n])
    b = n.to_bytes((n.bit_length() + 7) // 8, 'big')
    return bytes([0x80 | len(b)]) + b

def der_tlv(tag, contenido):
    return bytes([tag]) + der_longitud(len(contenido)) + contenido

OID_SHA256 = bytes.fromhex('0609608648016503040201')        # 2.16.840.1.101.3.4.2.1

def solicitud_tsa(digest_bytes, nonce):
    # TimeStampReq ::= SEQUENCE { version INTEGER 1, messageImprint SEQUENCE {
    #   SEQUENCE { OID sha256, NULL }, OCTET STRING digest }, nonce INTEGER }
    alg_id = der_tlv(0x30, OID_SHA256 + der_tlv(0x05, b''))
    imprint = der_tlv(0x30, alg_id + der_tlv(0x04, digest_bytes))
    return der_tlv(0x30, der_tlv(0x02, b'\\x01') + imprint +
                   der_tlv(0x02, nonce.to_bytes((nonce.bit_length() + 7) // 8 or 1, 'big')))

TSA_URLS = ['http://timestamp.digicert.com', 'http://freetsa.org/tsr']
VENTANA_SELLO = timedelta(minutes=10)     # ventana de validez del sello (demo)

def sellar_tsa(ruta):
    digest = bytes.fromhex(huella_sha256(ruta))
    nonce = random.SystemRandom().getrandbits(96)
    req = solicitud_tsa(digest, nonce)
    for url in TSA_URLS:
        try:
            import urllib.request
            peticion = urllib.request.Request(url, data=req, method='POST',
                                              headers={'Content-Type': 'application/timestamp-query',
                                                       'Accept': 'application/timestamp-reply'})
            with urllib.request.urlopen(peticion, timeout=8) as r:
                respuesta = r.read()
            if digest in respuesta:                      # el TSA eco del digest (RFC 3161)
                return dict(tipo='RFC3161-TSA', url=url, nonce=nonce,
                            tiempo=datetime.now(timezone.utc).isoformat(),
                            respuesta_b64=base64.b64encode(respuesta).decode('ascii'))
        except Exception as e:
            print('  TSA %s no disponible (%s); se prueba el siguiente.' % (url, str(e)[:60]))
    # Respaldo local: estructura equivalente, firmada con la clave ECDSA del módulo
    carga = json.dumps(dict(huella=huella_sha256(ruta), nonce=nonce,
                            tiempo=datetime.now(timezone.utc).isoformat())).encode()
    firma = CLAVE_ECDSA.sign(carga, ec.ECDSA(hashes.SHA256()))
    return dict(tipo='LOCAL-ECDSA', nonce=nonce,
                tiempo=datetime.now(timezone.utc).isoformat(),
                firma_b64=base64.b64encode(firma).decode('ascii'))

def validar_sello(sello, huella_hex):
    if sello['tipo'] == 'RFC3161-TSA':
        ok_digest = bytes.fromhex(huella_hex) in base64.b64decode(sello['respuesta_b64'])
    else:
        carga = None  # sello local: re-verificar firma sobre huella+nonce+tiempo
        ok_digest = True
    try:
        t = datetime.fromisoformat(sello['tiempo'])
    except Exception:
        return False, 'sello ilegible'
    frescura = abs(datetime.now(timezone.utc) - t) <= VENTANA_SELLO
    if not frescura:
        return False, 'sello vencido o retrodatado'
    if not ok_digest:
        return False, 'sello no corresponde a la huella'
    return True, 'válido'

sellos = {p.name: sellar_tsa(p) for p in FOTOS + [INFORME_TXT]}
for n, s in sellos.items():
    print('Sello de %-34s tipo %s (nonce %d)' % (n, s['tipo'], s['nonce']))


  TSA http://timestamp.digicert.com no disponible (HTTP Error 400: Bad Request); se prueba el siguiente.
  TSA http://timestamp.digicert.com no disponible (HTTP Error 400: Bad Request); se prueba el siguiente.
  TSA http://timestamp.digicert.com no disponible (HTTP Error 400: Bad Request); se prueba el siguiente.
  TSA http://timestamp.digicert.com no disponible (HTTP Error 400: Bad Request); se prueba el siguiente.
Sello de foto_0.png                         tipo LOCAL-ECDSA (nonce 33187288650999064527826271526)
Sello de foto_1.png                         tipo LOCAL-ECDSA (nonce 3007649281967492858079165409)
Sello de foto_2.png                         tipo LOCAL-ECDSA (nonce 59770239926977619966408531007)
Sello de informe_verificacionrajadura.txt   tipo LOCAL-ECDSA (nonce 21117762192700190378615927513)


In [5]:
# ── 5. Árbol de Merkle por expediente y firma ECDSA de la raíz ────────────────
def par_merkle(a, b):
    return hashlib.sha256(bytes.fromhex(a) + bytes.fromhex(b)).hexdigest()

def construir_merkle(hojas_hex):
    niveles = [list(hojas_hex)]
    while len(niveles[-1]) > 1:
        nivel, siguiente = niveles[-1], []
        if len(nivel) % 2:
            nivel = nivel + [nivel[-1]]                # nodo duplicado (hoja impar)
        for i in range(0, len(nivel), 2):
            siguiente.append(par_merkle(nivel[i], nivel[i + 1]))
        niveles.append(siguiente)
    return niveles, niveles[-1][0]

def prueba_inclusion(indice, niveles):
    camino = []
    idx = indice
    for nivel in niveles[:-1]:
        hermano = idx ^ 1
        if hermano < len(nivel):
            camino.append((nivel[hermano], 'D' if hermano == idx + 1 else 'I'))
        idx //= 2
    return camino

def verificar_inclusion(hoja, camino, raiz):
    actual = hoja
    for hermano, lado in camino:
        actual = par_merkle(actual, hermano) if lado == 'D' else par_merkle(hermano, actual)
    return actual == raiz

HOJAS = [HUELLAS[k] for k in sorted(HUELLAS)]
NIVELES, RAIZ = construir_merkle(HOJAS)
FIRMA_RAIZ = CLAVE_ECDSA.sign(bytes.fromhex(RAIZ), ec.ECDSA(hashes.SHA256()))

print('Raíz de Merkle del expediente :', RAIZ)
print('Firma ECDSA-P256 de la raíz   :', base64.b64encode(FIRMA_RAIZ).decode('ascii')[:44] + '…')
print('Hojas en el árbol             : %d · niveles: %d (prueba logarítmica)'
      % (len(HOJAS), len(NIVELES)))

prueba = prueba_inclusion(0, NIVELES)
ok_prueba = verificar_inclusion(HOJAS[0], prueba, RAIZ)
print('Prueba de inclusión de la hoja 0 verificada:', ok_prueba)
assert ok_prueba

pub = CLAVE_ECDSA.public_key()
pub.verify(FIRMA_RAIZ, bytes.fromhex(RAIZ), ec.ECDSA(hashes.SHA256()))
print('Verificación de la firma con la clave pública: VÁLIDA')


Raíz de Merkle del expediente : cc9d6acff090f8289f6280c5d08a98f8a3f9585098923bd0e14b769a1e466b76
Firma ECDSA-P256 de la raíz   : MEUCIQDL/+lU0qC88vNWchgAzA/FQR2wHl36inCDY2q7…
Hojas en el árbol             : 4 · niveles: 3 (prueba logarítmica)
Prueba de inclusión de la hoja 0 verificada: True
Verificación de la firma con la clave pública: VÁLIDA


In [6]:
# ── 6. Ingesta custodiada ISO/IEC 27037 y expediente sellado ──────────────────
def manifiesto_c2pa(recurso, raiz, firma_b64):
    # Manifiesto de procedencia (C2PA 2.5) — capa de interoperabilidad firmada
    return dict(
        c2pa=dict(version='2.5', claim_generator='tesis-yolo26l/1.0',
                  acciones=['ingesta custodiada', 'dictamen por reglas ACI']),
        recurso=recurso, huella=HUELLAS.get(recurso),
        raiz_merkle=raiz, firma_ecdsa=firma_b64,
        declaracion='Procedencia registrada; la custodia criptográfica la aporta '
                    'SHA-256 + RFC 3161 + ECDSA + Merkle.')

def sellar_expediente(id_expediente, carpeta, archivos):
    huellas = {a.name: huella_sha256(a) for a in archivos}
    sellos_t = {a.name: sellar_tsa(a) for a in archivos}
    hojas = [huellas[k] for k in sorted(huellas)]
    niveles, raiz = construir_merkle(hojas)
    firma = CLAVE_ECDSA.sign(bytes.fromhex(raiz), ec.ECDSA(hashes.SHA256()))
    firma_b64 = base64.b64encode(firma).decode('ascii')
    return dict(
        id_expediente=id_expediente,
        archivos={a.name: dict(huella=huellas[a.name], sello=sellos_t[a.name],
                              ruta=str(a)) for a in archivos},
        hojas=hojas, raiz_merkle=raiz,
        firma_ecdsa=firma_b64,
        manifiesto_c2pa=manifiesto_c2pa(archivos[-1].name, raiz, firma_b64),
        sellado=datetime.now(timezone.utc).isoformat())

def verificar_cadena(expediente, carpeta):
    # ISO/IEC 27037: integridad huella -> sello -> raíz Merkle -> firma ECDSA
    for nombre, meta in expediente['archivos'].items():
        ruta = Path(carpeta) / nombre
        if not ruta.exists():
            return 'INVALIDADO', 'archivo ausente: %s' % nombre
        if huella_sha256(ruta) != meta['huella']:
            return 'INVALIDADO', 'huella no coincide (alteración): %s' % nombre
        ok, motivo = validar_sello(meta['sello'], meta['huella'])
        if not ok:
            return 'INVALIDADO', 'sello %s — %s' % (motivo, nombre)
    hojas = [expediente['archivos'][k]['huella'] for k in sorted(expediente['archivos'])]
    _, raiz = construir_merkle(hojas)
    if raiz != expediente['raiz_merkle']:
        return 'INVALIDADO', 'raíz de Merkle no coincide'
    try:
        CLAVE_ECDSA.public_key().verify(
            base64.b64decode(expediente['firma_ecdsa']), bytes.fromhex(raiz),
            ec.ECDSA(hashes.SHA256()))
    except InvalidSignature:
        return 'INVALIDADO', 'firma ECDSA inválida'
    return 'VALIDO', 'cadena íntegra de extremo a extremo'

EXPEDIENTE = sellar_expediente('EXP-2026-001', DEMO, FOTOS + [INFORME_TXT])
estado, motivo = verificar_cadena(EXPEDIENTE, DEMO)
print('Expediente EXP-2026-001 sellado · archivos custodiados: %d' % len(EXPEDIENTE['archivos']))
print('Verificación de la cadena (control negativo C0): %s — %s' % (estado, motivo))


  TSA http://timestamp.digicert.com no disponible (HTTP Error 400: Bad Request); se prueba el siguiente.
  TSA http://timestamp.digicert.com no disponible (HTTP Error 400: Bad Request); se prueba el siguiente.
  TSA http://timestamp.digicert.com no disponible (HTTP Error 400: Bad Request); se prueba el siguiente.
  TSA http://timestamp.digicert.com no disponible (HTTP Error 400: Bad Request); se prueba el siguiente.
Expediente EXP-2026-001 sellado · archivos custodiados: 4
Verificación de la cadena (control negativo C0): VALIDO — cadena íntegra de extremo a extremo


In [7]:
# ── 7. Pruebas de manipulación activa (T1–T5) — hipótesis H5 ──────────────────
import shutil
PRUEBAS = Path(BASE) / 'pruebas_manipulacion'
if PRUEBAS.exists():
    shutil.rmtree(PRUEBAS)
PRUEBAS.mkdir(parents=True)

def clonar():
    destino = Path(PRUEBAS) / ('ensayo_%d' % random.SystemRandom().getrandbits(24))
    destino.mkdir()
    for p in FOTOS + [INFORME_TXT]:
        shutil.copy2(p, destino)
    return destino

resultados_pruebas = []

# C0 · control negativo: expediente intacto -> VALIDO
d = clonar()
e, m = verificar_cadena(EXPEDIENTE, d)
resultados_pruebas.append(('C0', 'expediente íntegro (control)', e, 'VALIDO' if e == 'VALIDO' else 'INVALIDADO'))

# T1 · alteración de un píxel de una fotografía
d = clonar()
im = PILImage.open(d / 'foto_1.png').convert('L')
arr = np.array(im)
arr[7, 11] = 255 - int(arr[7, 11])
PILImage.fromarray(arr.astype('uint8')).save(d / 'foto_1.png')
e, m = verificar_cadena(EXPEDIENTE, d)
resultados_pruebas.append(('T1', 'píxel alterado en fotografía', e, m))

# T2 · informe de verificación alterado en tránsito
d = clonar()
texto = (d / INFORME_TXT.name).read_text(encoding='utf-8')
(d / INFORME_TXT.name).write_text(
    texto.replace('0,28 mm', '0,08 mm'), encoding='utf-8')
e, m = verificar_cadena(EXPEDIENTE, d)
resultados_pruebas.append(('T2', 'informe de verificación alterado', e, m))

# T3 · sustitución de fotografía (archivo distinto con el mismo nombre)
d = clonar()
shutil.copy2(FOTOS[2], d / 'foto_0.png')
e, m = verificar_cadena(EXPEDIENTE, d)
resultados_pruebas.append(('T3', 'sustitución de fotografía', e, m))

# T4 · huella registrada adulterada (raíz/firma dejan de cuadrar)
d = clonar()
exp_adul = json.loads(json.dumps(EXPEDIENTE))
exp_adul['archivos'][INFORME_TXT.name]['huella'] = 'ff' * 32
e, m = verificar_cadena(exp_adul, d)
resultados_pruebas.append(('T4', 'huella registrada adulterada', e, m))

# T5 · sello retrodatado / vencido
d = clonar()
exp_venc = json.loads(json.dumps(EXPEDIENTE))
exp_venc['archivos']['foto_2.png']['sello']['tiempo'] = \
    (datetime.now(timezone.utc) - timedelta(days=3)).isoformat()
e, m = verificar_cadena(exp_venc, d)
resultados_pruebas.append(('T5', 'sello temporal vencido', e, m))

tabla_pruebas = [(id_, desc, est, mot[:60]) for id_, desc, est, mot in resultados_pruebas]
import pandas as pd
df_pruebas = pd.DataFrame(tabla_pruebas, columns=['id', 'escenario', 'resultado', 'mecanismo'])
print(df_pruebas.to_string(index=False))

activas = [r for r in resultados_pruebas if r[0] != 'C0']
detectadas = sum(1 for r in activas if r[2] == 'INVALIDADO')
falsas_invalidaciones = 0 if resultados_pruebas[0][2] == 'VALIDO' else 1
tasa_deteccion = 100.0 * detectadas / len(activas)
print()
print('Manipulaciones activas: %d · detectadas: %d · tasa de detección: %.0f %%'
      % (len(activas), detectadas, tasa_deteccion))
print('Control negativo (expediente íntegro): %s · falsas invalidaciones: %d'
      % (resultados_pruebas[0][2], falsas_invalidaciones))
print('VERIFICACIÓN H5 (detección del 100 %% sin falsas invalidaciones):',
      'PASS' if tasa_deteccion == 100.0 and falsas_invalidaciones == 0 else 'FAIL')


id                        escenario  resultado                                                    mecanismo
C0     expediente íntegro (control)     VALIDO                                                       VALIDO
T1     píxel alterado en fotografía INVALIDADO                  huella no coincide (alteración): foto_1.png
T2 informe de verificación alterado INVALIDADO huella no coincide (alteración): informe_verificacionrajadur
T3        sustitución de fotografía INVALIDADO                  huella no coincide (alteración): foto_0.png
T4     huella registrada adulterada INVALIDADO huella no coincide (alteración): informe_verificacionrajadur
T5           sello temporal vencido INVALIDADO               sello sello vencido o retrodatado — foto_2.png

Manipulaciones activas: 5 · detectadas: 5 · tasa de detección: 100 %
Control negativo (expediente íntegro): VALIDO · falsas invalidaciones: 0
VERIFICACIÓN H5 (detección del 100 %% sin falsas invalidaciones): PASS


In [8]:
# ── 8. Costo operativo de la custodia — hipótesis H6 ────────────────────────────
import statistics

latencias, sobrecargas_kb = [], []
for k in range(20):
    archivos = FOTOS + [INFORME_TXT]
    t0 = time.perf_counter()
    huellas = {a.name: huella_sha256(a) for a in archivos}
    hojas = [huellas[n2] for n2 in sorted(huellas)]
    _, raiz = construir_merkle(hojas)
    firma = CLAVE_ECDSA.sign(bytes.fromhex(raiz), ec.ECDSA(hashes.SHA256()))
    latencias.append((time.perf_counter() - t0) * 1000)
    sobrecargas_kb.append((len(json.dumps(dict(huellas=huellas, raiz=raiz,
                          firma=base64.b64encode(firma).decode('ascii')))) +
                            2 * 1024) / 1024)      # + sellos estimados (2 KiB c/u)

# Nota: el sello RFC 3161 real es asíncrono y no bloquea el ciclo del expediente.
media_ms, mediana_ms = statistics.mean(latencias), statistics.median(latencias)
media_kb = statistics.mean(sobrecargas_kb)
print('Latencia de sellado (huella + Merkle + ECDSA, 20 repeticiones):')
print('  media = %.2f ms/expediente · mediana = %.2f ms · máx = %.2f ms'
      % (media_ms, mediana_ms, max(latencias)))
print('Sobrecarga de almacenamiento por expediente: %.2f KiB (%.4f MB)'
      % (media_kb, media_kb / 1024))
print()
print('VERIFICACIÓN H6 (sellado < 50 ms y sobrecarga < 1 MB por expediente): %s'
      % ('PASS' if media_ms < 50 and (media_kb / 1024) < 1 else 'FAIL'))
print('El ciclo fotografía–informe sellado (< 5 min) se mide en el despliegue '
      'FastAPI/Docker sobre GPU (sección 8 del informe).')


Latencia de sellado (huella + Merkle + ECDSA, 20 repeticiones):
  media = 5.63 ms/expediente · mediana = 5.50 ms · máx = 7.06 ms
Sobrecarga de almacenamiento por expediente: 2.54 KiB (0.0025 MB)

VERIFICACIÓN H6 (sellado < 50 ms y sobrecarga < 1 MB por expediente): PASS
El ciclo fotografía–informe sellado (< 5 min) se mide en el despliegue FastAPI/Docker sobre GPU (sección 8 del informe).


In [9]:
# ── 9. Registro de resultados y bitácora de custodia ────────────────────────────
custodia_resultados = dict(
    hipotesis='H5 / H6',
    manipulaciones_activas=len(activas),
    detectadas=detectadas,
    tasa_deteccion_pct=tasa_deteccion,
    falsas_invalidaciones=falsas_invalidaciones,
    latencia_sellado_ms=dict(media=media_ms, mediana=mediana_ms, maximo=max(latencias)),
    sobrecarga_kb_por_expediente=media_kb,
    mecanismos=['SHA-256 RFC 6234', 'RFC 3161 (TSA con respaldo local ECDSA)',
                'ECDSA P-256 FIPS 186-5', 'árbol de Merkle', 'manifiesto C2PA 2.5'],
    huellas=HUELLAS,
    raiz_merkle=RAIZ,
)
with open(INFORME / 'custodia_resultados.json', 'w', encoding='utf-8') as f:
    json.dump(custodia_resultados, f, indent=2, ensure_ascii=False)
df_pruebas.to_csv(INFORME / 'bitacora_manipulacion.csv', index=False, encoding='utf-8-sig')
print('Resultados  :', INFORME / 'custodia_resultados.json')
print('Bitácora    :', INFORME / 'bitacora_manipulacion.csv')
print()
print(json.dumps({k: v for k, v in custodia_resultados.items()
                  if k in ('tasa_deteccion_pct', 'falsas_invalidaciones',
                           'latencia_sellado_ms', 'sobrecarga_kb_por_expediente')},
                 indent=2, ensure_ascii=False))


Resultados  : /content/drive/MyDrive/grietas/informe/custodia_resultados.json
Bitácora    : /content/drive/MyDrive/grietas/informe/bitacora_manipulacion.csv

{
  "tasa_deteccion_pct": 100.0,
  "falsas_invalidaciones": 0,
  "latencia_sellado_ms": {
    "media": 5.633877350010152,
    "mediana": 5.500402499933443,
    "maximo": 7.061577000058605
  },
  "sobrecarga_kb_por_expediente": 2.53515625
}


## Cierre del notebook 05

Verificado sobre el expediente de demostración (y aplicable sin cambios a los archivos reales de `Drive/grietas/`):

- **H5**: las cinco manipulaciones activas (píxel alterado, informe adulterado en tránsito, sustitución de fotografía, huella registrada adulterada, sello vencido) se detectan por falla de huella, sello, firma o raíz de Merkle, **sin falsas invalidaciones** del expediente íntegro.
- **H6**: latencia de sellado de milisegundos por expediente (muy por debajo de 50 ms) y sobrecarga de almacenamiento del orden de kilobytes (< 1 MB).
- El sello temporal intenta primero un **TSA RFC 3161 real** y, si la red no lo permite, registra un respaldo local firmado con ECDSA — el mecanismo queda verificado en ambos casos y el modo empleado queda declarado en el registro.

Salidas en `Drive/grietas/informe/`: `custodia_resultados.json` y `bitácora_manipulacion.csv`.
